# 📋 Multi-Agent Insurance Claim Orchestration

## Overview
The `orchestration.ipynb` notebook implements a **sophisticated multi-agent orchestration system** for insurance claim processing using **Microsoft Agent Framework** and **Azure OpenAI**. This notebook demonstrates advanced concurrent agent coordination to analyze insurance claims from multiple specialized perspectives simultaneously.

## 🏗️ Architecture & Components

### Core Technologies
- **Microsoft Agent Framework**: Modern agent orchestration framework for concurrent execution
- **Azure OpenAI**: Cloud-based AI model hosting and inference
- **Concurrent Orchestration**: Parallel execution of multiple AI agents using ConcurrentBuilder
- **Azure Cosmos DB Integration**: Real-time data access through custom plugins
- **Azure Identity**: Secure authentication for Azure services

### Agent Specializations
The system creates three specialized AI agents that work concurrently:

1. **🔍 Claim Reviewer Agent**
   - Validates claim documentation completeness
   - Analyzes damage assessments and cost estimates
   - Identifies inconsistencies or missing information
   - Provides VALID/QUESTIONABLE/INVALID determinations
   - Equipped with Cosmos DB plugin for data retrieval

2. **⚠️ Risk Analyzer Agent**
   - Detects fraud patterns and suspicious indicators
   - Assesses claim authenticity and credibility
   - Analyzes timing, amounts, and circumstances
   - Provides LOW/MEDIUM/HIGH risk assessments
   - Equipped with Cosmos DB plugin for historical analysis

3. **📋 Policy Checker Agent**
   - Validates coverage against policy terms
   - Interprets limits, deductibles, and exclusions
   - Handles multiple policy types (Auto, Commercial, Motorcycle, etc.)
   - Provides COVERED/NOT COVERED/PARTIAL COVERAGE determinations
   - Uses Azure AI Search for policy document analysis

## 🔧 Key Functions

### `create_specialized_agents()`
- Creates Azure OpenAI chat client with proper authentication
- Configures each agent with specialized instructions and capabilities
- Sets up Cosmos DB plugins for data-enabled agents
- Returns a collection of ready-to-use specialized agents

### `run_insurance_claim_orchestration()`
- **Concurrent Processing**: All three agents analyze claims simultaneously using ConcurrentBuilder
- **Intelligent Task Distribution**: Each agent receives specialized instructions
- **Real-time Data Access**: Agents can retrieve claim data using provided claim IDs
- **Comprehensive Reporting**: Generates unified analysis reports combining all agent outputs
- **Error Handling**: Robust exception management and resource cleanup
- **Progress Tracking**: Detailed logging of orchestration stages

## 🚀 Orchestration Flow

1. **Agent Creation**: Initializes three specialized insurance processing agents
2. **Concurrent Workflow Setup**: Creates parallel execution framework using ConcurrentBuilder
3. **Task Distribution**: Assigns specialized analysis tasks to each agent
4. **Parallel Execution**: All agents work simultaneously on their respective analyses
5. **Result Aggregation**: Collects and consolidates outputs from all agents via event stream
6. **Report Generation**: Creates comprehensive analysis report with all findings

## 📊 Output Format

The system generates a comprehensive **Insurance Claim Analysis Report** that includes:

- **Individual Agent Assessments**: Detailed analysis from each specialized perspective
- **Claim Validity Status**: Complete documentation and accuracy assessment
- **Risk Profile Analysis**: Fraud indicators and authenticity evaluation
- **Policy Coverage Determination**: Coverage eligibility and terms validation
- **Unified Recommendations**: Consolidated next steps based on all agent analyses

## 💡 Advanced Features

- **True Concurrent Execution**: Leverages Microsoft Agent Framework's ConcurrentBuilder for efficient parallelism
- **Database Integration**: Real-time access to claim and policy data through plugins
- **Flexible Configuration**: Environment-based model and endpoint configuration
- **Comprehensive Logging**: Detailed progress tracking and status updates
- **Event-based Results**: Streams results as they become available from each agent
- **Error Recovery**: Graceful handling of individual agent failures

## 🎯 Use Case Example

The notebook includes a practical example demonstrating the orchestration of claim analysis for a specific claim ID and policy number, showing how the system processes claim details and retrieves additional data to provide comprehensive multi-agent analysis.

This implementation represents a cutting-edge approach to insurance claim processing, leveraging the power of Microsoft Agent Framework's concurrent orchestration to provide thorough, multi-perspective analysis that would traditionally require multiple human experts working in sequence.


In [ ]:
# Import necessary libraries for Microsoft Agent Framework orchestrationimport asyncioimport osfrom typing import Dict, Anyfrom azure.identity import DefaultAzureCredential, AzureCliCredentialfrom agent_framework import ChatMessage, ConcurrentBuilderfrom agent_framework.azure import AzureOpenAIChatClient# Import the Cosmos DB toolsfrom agents.cosmos_tools import get_document_by_claim_idfrom dotenv import load_dotenvload_dotenv(override=True)  # This forces a reload of the .env fileasync def create_specialized_agents():    """Create our specialized insurance processing agents using Microsoft Agent Framework."""        print("🔧 Creating specialized insurance agents...")        # Get environment variables    # For Agent Framework, we use AzureOpenAIChatClient which connects to Azure OpenAI    # Try to use DefaultAzureCredential first, fall back to AzureCliCredential    try:        credential = DefaultAzureCredential()    except Exception as e:
        print(f"⚠️  DefaultAzureCredential failed: {str(e)}, falling back to AzureCliCredential")
        credential = AzureCliCredential()        # Create Azure OpenAI chat client    # Agent Framework uses environment variables or explicit configuration    chat_client = AzureOpenAIChatClient(credential=credential)        agents = {}        # Create Claim Reviewer Agent with Cosmos DB access    print("🔍 Creating Claim Reviewer Agent...")    claim_reviewer_agent = chat_client.create_agent(        instructions="""You are an expert Insurance Claim Reviewer Agent specialized in analyzing and validating insurance claims.         Your primary responsibilities include:        1. Use the get_document_by_claim_id function to retrieve claim data by claim_id, then:        2. Review all claim details (dates, amounts, descriptions).        3. Verify completeness of documentation and supporting evidence.        4. Analyze damage assessments and cost estimates for reasonableness.        5. Validate claim details against policy requirements.        6. Identify inconsistencies, missing info, or red flags.        7. Provide a detailed assessment with specific recommendations.        **Response Format**:        A short paragraph description if the CLAIM STATUS is: VALID / QUESTIONABLE / INVALID ; Analysis: Summary of findings by component; Any missing Info / Concerns: List of issues or gaps;        Next Steps: Clear, actionable recommendations        """,        name="ClaimReviewer",        tools=[get_document_by_claim_id]    )        # Create Risk Analyzer Agent with Cosmos DB access    print("⚠️ Creating Risk Analyzer Agent...")    risk_analyzer_agent = chat_client.create_agent(        instructions="""You are the Risk Analysis Agent. Your role is to evaluate the authenticity of insurance claims and detect potential fraud using available claim data.        Core Functions:        - Analyze historical and current claim data        - Identify suspicious patterns, inconsistencies, or anomalies        - Detect fraud indicators        - Assess claim credibility and assign a risk score        - Recommend follow-up actions if warranted        Assessment Guidelines:        - Use the get_document_by_claim_id function to access claim records        - Look for unusual timing, inconsistent descriptions, irregular amounts, or clustering        - Check for repeat claim behavior or geographic overlaps        - Assess the overall risk profile of each claim        Fraud Indicators to Watch For:        - Claims with irregular timing        - Contradictory or vague damage descriptions        - Unusual or repetitive claim amounts        - Multiple recent claims under same or related profiles        - Geographic or temporal clustering of incidents        Output Format:        - Risk Level: LOW / MEDIUM / HIGH        - Risk Analysis: Brief summary of findings        - Indicators: List of specific fraud signals (if any)        - Risk Score: 1–10 scale        - Recommendation: Investigate / Monitor / No action needed        Base all assessments strictly on the available claim data. Use structured reasoning and avoid assumptions beyond the data.        """,        name="RiskAnalyzer",        tools=[get_document_by_claim_id]    )        # Create Policy Checker Agent    print("📋 Creating Policy Checker Agent...")    policy_checker_agent = chat_client.create_agent(        instructions="""You are the Policy Checker Agent.        Your task is to summarize a policy based on policy number.        Instructions:        - Do not analyze claim details directly.        - Use your search tool to locate policy documents by policy number or policy type.        - Identify relevant exclusions, limits, and deductibles.        - Base your determination only on the contents of the retrieved policy.        Output Format:        - Policy Number: [Policy number]        - Main important details        - Reference and quote specific policy sections that support your determination.        - Clearly explain how the policy language leads to your conclusion.        Be precise, objective, and rely solely on the policy content.        """,        name="PolicyChecker",    )        agents = {        'claim_reviewer': claim_reviewer_agent,        'risk_analyzer': risk_analyzer_agent,        'policy_checker': policy_checker_agent    }        print("✅ All specialized agents created successfully!")    return agents, chat_clientasync def run_insurance_claim_orchestration(claim_id: str, policy_number: str):    """Orchestrate multiple agents to process an insurance claim concurrently using Microsoft Agent Framework."""        print(f"🚀 Starting Concurrent Insurance Claim Processing Orchestration")    print(f"{'='*80}")        # Create our specialized agents    agents, chat_client = await create_specialized_agents()        # Create concurrent orchestration with all three agents    workflow = ConcurrentBuilder().participants([        agents['claim_reviewer'],        agents['risk_analyzer'],        agents['policy_checker']    ]).build()        try:                # Create task that instructs agents to retrieve claim details first        task = f"""Analyze the insurance claim with ID: {claim_id} or the policy number {policy_number} and come back with a critical solution for if the claim should be approved.CRITICAL: ALL AGENTS MUST USE THEIR AVAILABLE TOOLS TO RETRIEVE INFORMATIONAGENT-SPECIFIC INSTRUCTIONS:Claim Reviewer Agent: - MUST USE: get_document_by_claim_id("{claim_id}") to retrieve claim details- Review all claim documentation and assess completeness- Validate damage estimates and repair costs against retrieved data- Check for proper evidence and documentation in the claim data- Cross-reference claim amounts with industry standards- Provide VALID/QUESTIONABLE/INVALID determination with detailed reasoningRisk Analyzer Agent:- MUST USE: get_document_by_claim_id("{claim_id}") to retrieve claim data- Analyze the retrieved data for fraud indicators and suspicious patterns- Assess claim authenticity and credibility based on actual claim details- Check for unusual timing, amounts, or circumstances in the data- Look for inconsistencies between different parts of the claim- Provide LOW/MEDIUM/HIGH risk assessment with specific evidencePolicy Checker Agent (policy_checker_agent):- YOU DO NOT NEED TO LOOK INTO CLAIMS!- MUST USE: Your search capabilities to find relevant policy documents by policy number ("{policy_number}") or type found in the claim data- Search for policy documents using policy numbers- Identify relevant exclusions, limits, or deductibles from actual policy documents- Provide COVERED/NOT COVERED/PARTIAL COVERAGE determination with policy references- Quote specific policy sections that support your determinationIMPORTANT: Each agent MUST actively use their tools to retrieve and analyze actual data. Do not provide generic responses - base your analysis on the specific claim data and policy documents retrieved through your tools."""                # Run the concurrent orchestration        print(f"\n🔄 Invoking concurrent orchestration...")        events = await workflow.run(task)                print(f"\n🎉 All agents completed their analysis!")        print(f"{'─'*60}")                # Get outputs from the workflow        outputs = events.get_outputs()                # Collect results from all agents        results = []        if outputs:            for output in outputs:                # Output is a list of ChatMessage objects                messages = output if isinstance(output, list) else [output]                for msg in messages:                    if hasattr(msg, 'text') and msg.text:                        results.append(msg.text)                        author = getattr(msg, 'author_name', 'Agent')                        print(f"\n🤖 {author} Analysis:")                        print(f"{'─'*40}")                        print(msg.text)                # Create comprehensive analysis report        comprehensive_analysis = f"""{chr(10).join([f"### Agent Assessment:{chr(10)}{chr(10)}{result}{chr(10)}" for result in results])}"""                print(f"\n✅ Concurrent Insurance Claim Orchestration Complete!")        return comprehensive_analysis            except Exception as e:        print(f"❌ Error during orchestration: {str(e)}")        import traceback        traceback.print_exc()        raise            finally:        print(f"\n🧹 Orchestration cleanup complete.")

In [ ]:
claim_id = "CL001"
policy_number = "LIAB-AUTO-001"  # User provides the specific policy number to search
result = await run_insurance_claim_orchestration(claim_id, policy_number)